In [ ]:
# LLM Parser

ToDo:
- dynamisch für ganzen Data verzeichnis
- JSON abspeichern
- required im inneren objekten


Notizen:
- Personenverzeichnis: links: Rolle, Rechts: Schauspieler

SyntaxError: invalid syntax (2275760556.py, line 3)

In [7]:
from pathlib import Path
from openai import OpenAI

# connect to LM Studio
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

ocr_text = Path("../../data/02_ocr/R_9346_I_1.txt").read_text(encoding="utf-8")

json = {
  "type": "object",
  "properties": {
    "prüf-nr": {
      "type": "integer"
    },
    "Gesetz": {
      "type": "string"
    },
    "Ursprungs-Firma": {
      "type": "string"
    },
    "Titel des Bildes": {
      "type": "string"
    },
    "Text unter Titel des Bildes (Kurzbeschreibung des Filmes)": {
      "type": "string"
    },
    "Spielleitung / Personen der Handlung:": {
      "type": "string"
    },
    "Photographie": {
      "type": "string"
    },
    "Archiketur": {
      "type": "string"
    },
    "Personenverzeichnis": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "name": {
            "type": "string"
          },
          "rolle": {
            "type": "string"
          }
        },
        "required": [
          "name",
          "rolle"
        ]
      }
    },
    "Text": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "akt": {
            "type": "integer"
          },
          "text": {
            "type": "string"
          }
        },
        "required": [
          "akt",
          "text"
        ]
      }
    },
    "Auschnitte": {
      "type": "string"
    },
    "Länge": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "akt": {
            "type": "integer"
          },
          "meter": {
            "type": "integer"
          },
          "nach kürzung": {
            "type": "integer"
          },
          "gesamtlänge": {
            "type": "array",
            "items": {
              "type": "object",
              "properties": {
                "gesamtlänge": {
                  "type": "integer"
                },
                "nach kürzung": {
                  "type": "integer"
                }
              }
              # TODO: required für das innere Array hier einfügen, falls nötig
            }
          }
        }
      }
    },
    "Entscheidung": {
      "type": "string"
    },
    "ort und datum": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "ort": {
            "type": "string"
          },
          "datum": {
            "type": "string"
          }
        },
        "required": [
          "ort",
          "datum"
        ]
      }
    },
    "Filmprüfstelle": {
      "type": "string"
    }
  },
  "required": [
      "prüf-nr",
      "Gesetz",
      "Ursprungs-Firma",
      "Titel des Bildes",
      "Text unter Titel des Bildes (Kurzbeschreibung des Filmes)",
      "Spielleitung / Personen der Handlung:",
      "Photographie",
      "Archiketur",
      "Personenverzeichnis",
      "Text",
      "Auschnitte",
      "Länge",
      "Entscheidung",
      "Filmprüfstelle"
  ]
}
    

response = client.chat.completions.create(
    model="local-model",
    response_format={
        "type": "json_schema", 
        "json_schema": {
            "name": "ocr_extraction",
            "schema": json,
            "strict": True
        }
    },
    messages=[
        {"role": "system", 
         "content": # You are a helpful assistant that extracts structured Output. Extract the OCR text into a structured format
            """ Du bist ein hochpräzises Daten-Extraktions-System. 
                Deine einzige Aufgabe ist es, Informationen aus einem unstrukturierten, teils fehlerhaften OCR-Text zu extrahieren und in ein strikt vorgegebenes JSON-Schema zu überführen.

                REGELN FÜR DIE EXTRAKTION:
                1. OCR-Korrektur: Der Eingabetext enthält typische OCR-Fehler (falsche Zeichen, verdrehte Reihenfolge, fehlende Leerzeichen). Analysiere den semantischen Kontext und korrigiere offensichtliche Zeichenfehler automatisch.
                2. Strikte Faktentreue: Erfinde NIEMALS Informationen hinzu. Leite keine Daten ab, die nicht explizit oder als klarer OCR-Fehler im Text stehen.
                3. Fehlende Werte: Wenn eine Information für ein Feld des JSON-Schemas im Text nicht auffindbar ist, setze den Wert ZWINGEND auf `null`. Verwende keine Platzhalter wie "unbekannt" oder "N/A".
                4. Relevanz-Filter: Der OCR-Text enthält wahrscheinlich irrelevante Kopfzeilen, Fußzeilen oder Zusatzinformationen. Ignoriere alles, was nicht im Ziel-Schema abgefragt wird.
                5. Formatierung: Halte dich exakt an die Datentypen des Schemas (Zahlen als Number, nicht als String; Datumsformate exakt wie gefordert).

                Gib AUSSCHLIESSLICH das finale JSON-Objekt aus. Keine Erklärungen, kein Markdown-Codeblock, keine einleitenden Worte."""},
        {"role": "user", "content": ocr_text}
    ],
    temperature=0.3
)

output_text = response.choices[0].message.content

print(output_text)

{
  "prüf-nr": 9346,
  "Gesetz": "Reichs-Strafgesetzbuch",
  "Ursprungs-Firma": "Solar-Film G. m. b. H.",
  "Titel des Bildes": "Lady Florence... Ressel era",
  "Text unter Titel des Bildes (Kurzbeschreibung des Filmes)": "Sensationsschauspiel in 5 Akten von Jos. Delmont",
  "Spielleitung / Personen der Handlung:": "Rolf Brunner, August Brückner, Alfred Columbu5, M Wolle, Lady Florence, Lord William, Heinrich Peer, Lord Henry Fred Selva-Goebels, Walter Formes, Dr. Munson, Heinz Burkart, Jack PaulPassarge",
  "Photographie": "August Brückner",
  "Archiketur": "Alfred Columbu5",
  "Personenverzeichnis": [
    {
      "name": "Lady Florence",
      "rolle": "Hauptfigur/Charakter"
    },
    {
      "name": "Lord William",
      "rolle": "Heinrich Peer"
    }
  ],
  "Text": [
    {
      "akt": 1,
      "text": "Der Jrre. Lord Henrh (Fred SelVa-Goebel). [...] Ende des 1. Aktes."
    },
    {
      "akt": 2,
      "text": "Dr. Munson ist auf Schloß Reedcliffe eingetroffen. [...] Ende des 2.